In [ ]:
import base64
import gzip
import json
import pickle
import pyrender
import numpy as np
import random
import secrets
import trimesh
import voxeloo
import zlib

from PIL import Image
from dataclasses import dataclass
from functools import lru_cache
from matplotlib import pyplot as plt
from scipy.ndimage import (
    distance_transform_edt as edt,
    gaussian_filter as blur, 
)
from typing import List, Optional, Dict, Tuple
from scipy import ndimage

### Pipeline Input

In [ ]:
CHUNK_PADDING = 64
CHUNK_DIM = (2048, 2048)
CHUNK_ORIGIN = (-2048, -2048)

In [ ]:
MAP_DIM = (CHUNK_DIM[0] + 2 * CHUNK_PADDING, CHUNK_DIM[1] + 2 * CHUNK_PADDING)
MAP_ORIGIN = (CHUNK_ORIGIN[0] - CHUNK_PADDING, CHUNK_ORIGIN[1] - CHUNK_PADDING)

### Visualization routines

In [ ]:
def heatmap_to_image(heat):
    return Image.fromarray(np.uint8(heat * 255) , "L")

In [ ]:
def hex_to_rgb(value):
    return [
        (value >> 24) & 0xff,
        (value >> 16) & 0xff,
        (value >> 8) & 0xff,
    ]

In [ ]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

### Noise routines

In [ ]:
def seed_hash(key):
    return np.uint32(zlib.adler32(str(key).encode("utf-8")))

In [ ]:
def domain(shape, scale):
    coords = np.meshgrid(
        *[np.arange(dim) for dim in shape],
        indexing="ij",
    )
    if len(shape) == 2:
        coords[0] += MAP_ORIGIN[0]
        coords[1] += MAP_ORIGIN[1]
    else:
        coords[0] += MAP_ORIGIN[0]
        coords[1] += MAP_ORIGIN[1]
        coords[2] += MAP_ORIGIN[2]   
    return tuple(coord / scale for coord in coords)[::-1]

In [ ]:
@lru_cache(maxsize=None) 
def gen(seed=None):
    return voxeloo.noise.SimplexNoise(seed_hash(seed or 1))
    
def noise(coords, seed=None):
    return gen(seed).noise(coords.reshape(-1, coords.shape[-1])).reshape(*coords.shape[0:-1])

In [ ]:
def fractal_noise(shape, period, octaves, persistence=0.5, lacunarity=2.0, transform=None, seed=None):
    ret = np.zeros(shape, dtype=float)
    for i in range(octaves):
        coords = np.stack(domain(shape, period / lacunarity**i), axis=-1)
        if transform:
            coords = transform(coords)
        ret += persistence**i * noise(coords, seed)
    return ret

In [ ]:
def explicit_noise(shape, period, weights, lacunarity=2.0, transform=None, seed=None):
    ret = np.zeros(shape, dtype=float)
    for i, w in enumerate(weights):
        coords = np.stack(domain(shape, period / lacunarity**i), axis=-1)
        if transform:
            coords = transform(coords)
        ret += w * noise(coords, seed)
    return ret

In [ ]:
def linear_boundary(mask, radius):
    ret = edt(mask)
    ret[mask] = np.clip((ret - radius) / (1 - radius), 0, 1)[mask]
    return 1 - ret

In [ ]:
def smooth_buckets(shape, buckets, period=32):
    nm = explicit_noise(shape, 128, [1, 0, 1, 1, 0.4, 0.2])
    thresholds = np.quantile(nm, np.linspace(0, 1, buckets + 1)[1:])

    buckets = np.zeros(shape, dtype=int)
    for i, t in reversed(list(enumerate(thresholds))):
        buckets[nm < t] = i
    return buckets

In [ ]:
def masked_blur(x, mask, radius):
    mask = mask.astype(float)
    return blur(mask * x, radius) / (blur(mask, radius) + 0.0001)

In [ ]:
%%time 

x = explicit_noise(MAP_DIM, 2048, [0.1, 2, 4, 1], seed="roads")
x = linear_boundary(np.abs(x) < 0.1, 8)
heatmap_to_image((x - x.min()) / (x.max() - x.min()))

In [ ]:
%%time 

x = np.exp(explicit_noise(MAP_DIM, 1024, [8, 4, 2, 1], seed=5))
x += 40 * np.abs(fractal_noise(MAP_DIM, 64, octaves=3, persistence=0.4, seed=5))
x = linear_boundary(x > 80, 64)
heatmap_to_image((x - x.min()) / (x.max() - x.min()))

In [ ]:
buckets = smooth_buckets(MAP_DIM, 2)
Image.fromarray(
    np.random.randint(0, 255, size=(buckets.max() + 1, 3), dtype=np.uint8)[buckets]
)

### Define the world map types

In [ ]:
@dataclass
class RegionMap:
    ids: Dict[str, int]
    assignments: np.ndarray
    colors: List[int]
    radii: List[int]
        
@dataclass
class HeightMap:
    data: np.ndarray
        
@dataclass
class Column:
    name: str
    samples: List[np.ndarray]
    color: int

@dataclass
class SurfaceMap:
    ids: Dict[str, int]
    assignments: np.ndarray
    columns: List[Column]
        
@dataclass
class ColumnMap:
    data: np.ndarray

### Define the different regions

In [ ]:
class Mountains:
    
    name = "mountains"
    color = 0xdadfe3ff
    radius = 8
    
    @staticmethod
    def add_region(rm: RegionMap):
        nm = np.exp(explicit_noise(rm.assignments.shape, 1024, [8, 4, 2, 1], seed="mountains"))
        nm = nm + np.abs(explicit_noise(rm.assignments.shape, 64, [40, 20], seed="mountains"))
        mask = np.logical_and(nm > 80.0, rm.assignments == 0)
        rm.assignments[mask] = rm.ids["mountains"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        mask = rm.assignments == rm.ids["mountains"]
        wm = linear_boundary(mask, 128)
        nm = explicit_noise(hm.data.shape, 256, [32, 48, 32, 16, 8, 4])
        hm.data[mask] += wm[mask] * (nm[mask] + 128.0)
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap):
        mask = rm.assignments == rm.ids["mountains"]
        dm = edt(mask)
        
        # Default to snow
        sm.assignments[mask] = sm.ids["snow_peak"]

        # Add stone to the mid area
        nm = 16 * explicit_noise(hm.data.shape, 128, [1, 0, 1, 1, 0.4, 0.2], seed=1)
        stone_mask = np.logical_or(dm < 80 + nm, hm.data < 110 + nm)
        sm.assignments[np.logical_and(mask, stone_mask)] = sm.ids["stone"]
        
        # Add dirt to the low area
        nm = 10 + 8 * explicit_noise(hm.data.shape, 128, [1, 0, 1, 1, 0.4, 0.2], seed=2)
        sm.assignments[np.logical_and(mask, dm < nm)] = sm.ids["dirt"]

In [ ]:
class Canyons:
    
    name = "canyons"
    color = 0x574c43ff
    radius = 2
    
    @staticmethod
    def add_region(rm: RegionMap):
        nm = fractal_noise(
            rm.assignments.shape,
            1024,
            6,
            persistence=0.45,
            lacunarity=2.0,
            seed="canyons",
        )
        mask = np.logical_and(np.abs(nm) < 0.04, rm.assignments == 0)
        rm.assignments[mask] = rm.ids["canyons"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        nm = fractal_noise(hm.data.shape, 32, octaves=4, persistence=0.5)
        mask = rm.assignments == rm.ids["canyons"]
        hm.data[mask] += 4.0 * nm[mask] - 16.0
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap):
        mask = rm.assignments == rm.ids["canyons"]
        sm.assignments[mask] = sm.ids["canyon_bed"]

In [ ]:
class Ridges:
    
    name = "ridges"
    color = 0x8f7f57ff
    radius = 8
    
    @staticmethod
    def add_region(rm: RegionMap):
        nm = explicit_noise(
            rm.assignments.shape,
            1024,
            [2.0, 0.0, 1.0, 1.0, 0.3, 0.2, 0.05],
            seed="ridges"
        )
        mask = np.logical_and(nm > 0.7, rm.assignments == 0)
        rm.assignments[mask] = rm.ids["ridges"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        mask = rm.assignments == rm.ids["ridges"]
        wm = linear_boundary(mask, 64)
        nm = explicit_noise(hm.data.shape, 128, [8, 32, 16, 8, 4, 4])
        hm.data[mask] += wm[mask] * (nm[mask] + 32.0)
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap):
        mask = rm.assignments == rm.ids["ridges"]
        dm = edt(mask) 
        
        # Default to stone
        sm.assignments[mask] = sm.ids["stone"]

        # Add stone to the mid area
        nm = 16 * explicit_noise(hm.data.shape, 128, [1, 0, 1, 1, 0.4, 0.2], seed=1)
        stone_mask = np.logical_or(dm < 80 + nm, hm.data < 40 + nm)
        sm.assignments[np.logical_and(mask, stone_mask)] = sm.ids["dirt"]
        
        # Add grass to the low area
        nm = 10 + 8 * explicit_noise(hm.data.shape, 128, [1, 0, 1, 1, 0.4, 0.2], seed=2)
        sm.assignments[np.logical_and(mask, dm < nm)] = sm.ids["grass"]        

In [ ]:
class Escarpment:
    
    name = "escarpment"
    color = 0xc4d66bff
    radius = 2
    
    @staticmethod
    def add_region(rm: RegionMap):
        nm = explicit_noise(rm.assignments.shape, 256, [1.0, 0.5], seed="escarpment")
        mask = np.logical_and(nm > 0.5, rm.assignments == 0)
        rm.assignments[mask] = rm.ids["escarpment"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        nm = fractal_noise(hm.data.shape, 64, octaves=3, persistence=0.2)
        mask = rm.assignments == rm.ids["escarpment"]
        hm.data[mask] += 8 + 8 * nm[mask]
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap):
        shape = rm.assignments.shape
        mask = rm.assignments == rm.ids["escarpment"]
        
        # Default to grass 
        sm.assignments[mask] = sm.ids["grass"]   
        
        # Add some patches of stone
        nm = explicit_noise(shape, 256, [0.1, 0.5, 0.5, 0.5, 0.2, 0.1], seed="stone")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["stone"]

In [ ]:
class Hills:
    
    name = "hills"
    color = 0x70ba6cff
    radius = 16
    
    @staticmethod
    def add_region(rm: RegionMap):
        nm = explicit_noise(rm.assignments.shape, 256, [1.0, 0.5], seed="hills")
        mask = np.logical_and(nm > 0.1, rm.assignments == 0)
        rm.assignments[mask] = rm.ids["hills"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        mask = rm.assignments == rm.ids["hills"]
        wm = linear_boundary(mask, 16)
        nm = fractal_noise(hm.data.shape, 64, octaves=3, persistence=0.3)
        hm.data[mask] += wm[mask] * (4 + 16 * nm[mask])
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap):
        shape = rm.assignments.shape
        mask = rm.assignments == rm.ids["hills"]
        
        # Default to grass 
        sm.assignments[mask] = sm.ids["grass"]   
        
        # Add some patches of stone
        nm = explicit_noise(shape, 256, [0.1, 0.5, 0.5, 0.5, 0.2, 0.1], seed="stone")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["stone"]

        # Add some patches of moss
        nm = explicit_noise(shape, 256, [2, 1.5, 1, 0.2], seed="moss")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["moss"]

        # Add some patches of grass_on_clay
        nm = explicit_noise(shape, 256, [2, 1.5, 1, 0.2], seed="grass_on_clay")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["grass_on_clay"]

In [ ]:
class Knolls:
    
    name = "knolls"
    color = 0x458f49ff
    radius = 16
    
    @staticmethod
    def add_region(rm: RegionMap):
        nm = explicit_noise(rm.assignments.shape, 128, [1.0, 0.5], seed="knolls")
        mask = np.logical_and(nm > 0.1, rm.assignments == 0)
        rm.assignments[mask] = rm.ids["knolls"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        nm = fractal_noise(hm.data.shape, 256, octaves=3, persistence=0.3)
        mask = rm.assignments == rm.ids["knolls"]
        hm.data[mask] += 8 * nm[mask]
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap):
        shape = rm.assignments.shape
        mask = rm.assignments == rm.ids["knolls"]
        
        # Default to grass 
        sm.assignments[mask] = sm.ids["grass"]   
        
        # Add some patches of stone
        nm = explicit_noise(shape, 256, [0.1, 0.5, 0.5, 0.5, 0.2, 0.1], seed="stone")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["stone"]
        
        # Add some patches of stone
        nm = explicit_noise(shape, 256, [0.3, 0.3, 0.4, 0.7, 0.1], seed="dirt")
        sm.assignments[np.logical_and(mask, nm > 0.5)] = sm.ids["dirt"]

        # Add some patches of moss
        nm = explicit_noise(shape, 256, [2, 1.5, 1, 0.2], seed="moss")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["moss"]
        
        # Add some patches of grass_on_clay
        nm = explicit_noise(shape, 256, [2, 1.5, 1, 0.2], seed="grass_on_clay")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["grass_on_clay"]

In [ ]:
class Plains:
    
    name = "plains"
    color = 0x4dc465ff
    radius = 16
    
    @staticmethod
    def add_region(rm: RegionMap):
        rm.assignments[rm.assignments == 0] = rm.ids["plains"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        pass
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap):
        shape = rm.assignments.shape
        mask = rm.assignments == rm.ids["plains"]
        
        # Default to grass 
        sm.assignments[mask] = sm.ids["grass"]   
        
        # Add some patches of dirt
        nm = explicit_noise(shape, 256, [0.3, 0.3, 0.4, 0.7, 0.1], seed="dirt")
        sm.assignments[np.logical_and(mask, nm > 0.5)] = sm.ids["dirt"]

        # Add some patches of moss
        nm = explicit_noise(shape, 256, [2, 1.5, 1, 0.2], seed="moss")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["moss"]
        
        # Add some patches of grass_on_clay
        nm = explicit_noise(shape, 256, [2, 1.5, 1, 0.2], seed="grass_on_clay")
        sm.assignments[np.logical_and(mask, nm > 0.6)] = sm.ids["grass_on_clay"]

In [ ]:
regions = [
    Mountains(),
    Canyons(),
    Ridges(),
    Escarpment(),
    Hills(),
    Knolls(),
    Plains(),
]

### Generate the region maps

In [ ]:
rm = RegionMap(
    ids={"none": 0, **{r.name: i + 1 for i, r in enumerate(regions)}},
    assignments=np.zeros(shape=MAP_DIM, dtype=int),
    colors=[[0, 0, 0], *[hex_to_rgb(r.color) for r in regions]],
    radii=[1, *[r.radius for r in regions]],
)

In [ ]:
%%time

for region in regions:
    region.add_region(rm)

In [ ]:
for name, i in rm.ids.items():
    print(f"{name}: {(rm.assignments == i).mean()}")

In [ ]:
Image.fromarray(np.array(rm.colors, dtype=np.uint8)[rm.assignments])

### Generate the height maps

In [ ]:
hm = HeightMap(
    data=explicit_noise(MAP_DIM, 1024, [32, 16, 8, 4])
)

In [ ]:
%%time

for region in regions:
    region.add_heights(rm, hm)

In [ ]:
def boundary_mask(mask, radius):
    ret = edt(mask)
    ret[mask] = np.clip((ret - radius) / (1 - radius - 1e-6), 0, 1)[mask]
    return ret

weight = np.zeros(rm.assignments.shape)
for name, i in rm.ids.items():
    weight += boundary_mask(rm.assignments == i, rm.radii[i])
heatmap_to_image(weight)

In [ ]:
%%time 

for iteration in range(8):
    hm.data = (1 - weight) * hm.data + weight * blur(hm.data, 1)

In [ ]:
plt.hist(hm.data.flatten())
plt.show()

In [ ]:
heatmap_to_image(
    (hm.data - hm.data.min()) / (hm.data.max() - hm.data.min())
)

In [ ]:
%%time

heights = (hm.data - hm.data.min() + 1).astype(int)
voxels = np.zeros(shape=(MAP_DIM[0], heights.max(), MAP_DIM[1]), dtype=bool)

for z in range(MAP_DIM[0]):
    for x in range(MAP_DIM[1]):
        h = heights[z, x]
        voxels[z, :h, x] = True

In [ ]:
#visualize_voxels(voxels[:1024, :, :1024])

### Generate the surface map

In [ ]:
@dataclass
class TerrainVoxel:
    name: str
    index: int
    color: int

In [ ]:
# These IDs match the terrain IDs in: src/galois/js/assets/blocks.ts
terrain = [
    TerrainVoxel("none", 0, 0x00000000),
    TerrainVoxel("basalt", 8, 0x332c2bff),
    TerrainVoxel("bedrock", 6, 0x4e4a54ff),
    TerrainVoxel("birch_log", 12, 0xd4cdc5ff),
    TerrainVoxel("clay", 17, 0x6e554fff),
    TerrainVoxel("coal_ore", 19, 0x282e2aff),
    TerrainVoxel("cobblestone", 5, 0x524535ff),
    TerrainVoxel("diamond_ore", 23, 0x89c5d9ff),
    TerrainVoxel("dirt", 2, 0x543919ff),
    TerrainVoxel("gold_ore", 11, 0xdecb52ff),
    TerrainVoxel("granite", 13, 0x636d70ff),
    TerrainVoxel("grass", 1, 0x319436ff),
    TerrainVoxel("gravel", 36, 0xab9865ff),
    TerrainVoxel("hay", 35, 0xa39f33ff),
    TerrainVoxel("limestone", 9, 0xc3c4b1ff),
    TerrainVoxel("moss", 40, 0x30784eff),
    TerrainVoxel("neptunium_ore", 30, 0x1d4030ff),
    TerrainVoxel("oak_log", 3, 0x4d442cff),
    TerrainVoxel("pumpkin", 10, 0xc78e08ff),
    TerrainVoxel("quartzite", 7, 0xa89c8aff),
    TerrainVoxel("rubber_log", 14, 0x94570cff),
    TerrainVoxel("silver_ore", 25, 0xd8e3e3ff),
    TerrainVoxel("stone", 4, 0xc1c9c8ff),
    TerrainVoxel("snow", 37, 0xf5f5f5ff),
]

terrain_index = {tv.name: tv.index for tv in terrain}

In [ ]:
sm = SurfaceMap(
    ids={},
    assignments=np.zeros(MAP_DIM, dtype=int),
    columns=[],
)

In [ ]:
def to_column(*names):
    return np.array([terrain_index[name] for name in names], dtype=int)

sm.columns.append(
    Column(
        name="grass",
        color=0x3cb556ff,
        samples=[
            to_column("grass", *["dirt"] * i) for i in [5, 7, 8, 9, 10]
        ],
    )
)

sm.columns.append(
    Column(
        name="moss",
        color=0x2c5c36ff,
        samples=[
            to_column("moss", *["dirt"] * i) for i in [5, 6, 7, 8, 9, 10]
        ],
    )
)

sm.columns.append(
    Column(
        name="dirt",
        color=0x634d2eff,
        samples=[
            to_column(*["dirt"] * i, *["stone"] * j)
            for i, j in [(5, 2), (6, 4), (7, 3), (8, 4)]
        ],
    )
)

sm.columns.append(
    Column(
        name="road",
        color=0x806d53ff,
        samples=[
            to_column(*["gravel"] * i, *["dirt"] * j) 
            for i, j in [(1, 5), (2, 4), (3, 7), (3, 8)]
        ],
    )
)

sm.columns.append(
    Column(
        name="stone",
        color=0xafb7bdff,
        samples=[
            to_column(*["stone"] * i) for i in [4, 5]
        ],
    )
)

sm.columns.append(
    Column(
        name="snow_peak",
        color=0xebf2f7ff,
        samples=[
            to_column(*["snow"] * 3, *["stone"] * i) for i in [3, 4]
        ],
    )
)

sm.columns.append(
    Column(
        name="grass_on_clay",
        color=0x6e554fff,
        samples=[
            to_column("grass", *["dirt"] * i, *["clay"] * j)
            for i, j in [(2, 3), (3, 5), (3, 7), (3, 8)]
        ],
    )
)

sm.columns.append(
    Column(
        name="canyon_bed",
        color=0x656e6dff,
        samples=[
            *[to_column(*["stone"] * i) for i in [3, 4, 5, 7]],
            *[to_column(*["granite"] * i) for i in [3, 4, 5, 7]],
        ],
    )
)

sm.ids = {column.name: i for i, column in enumerate(sm.columns)}

In [ ]:
%%time

for region in regions:
    region.add_surface(rm, hm, sm)

In [ ]:
colors = np.array(
    [hex_to_rgb(column.color) for column in sm.columns],
    dtype=np.uint8,
)
Image.fromarray(colors[sm.assignments])

### Add roads

In [ ]:
class Roads:
    
    name = "roads"
    color = 0xb3945fff
    radius = 2
    
    @staticmethod
    def add_region(rm: RegionMap):
        nm1 = explicit_noise(rm.assignments.shape, 2048, [2, 2, 4, 0.5, 0.1, 0.1], seed="roads_1")
        nm2 = explicit_noise(rm.assignments.shape, 2048, [1, 2, 4, 0.5, 0.1, 0.1], seed="roads_2")
        mm = np.logical_or(np.abs(nm1) < 0.01, np.abs(nm2) < 0.01)
        mask = np.logical_and(
            blur(mm.astype(float), 4) > 0.04,
            np.logical_not(rm.assignments == rm.ids["canyons"]),
        )
        rm.assignments[mask] = rm.ids["roads"]
    
    @staticmethod
    def add_heights(rm: RegionMap, hm: HeightMap):
        mask = rm.assignments == rm.ids["roads"]
        base = explicit_noise(rm.assignments.shape, 1024, [32, 16, 8, 4])
        rm = blur(hm.data + 0.1 * mask * (base - hm.data), 8)
        rm = masked_blur(rm, mask, 32)
        wm = linear_boundary(mask, 4)
        hm.data[mask] += wm[mask] * (rm[mask] - hm.data[mask])
    
    @staticmethod
    def add_surface(rm: RegionMap, hm: HeightMap, sm: SurfaceMap): 
        mask = np.logical_and(
            rm.assignments == rm.ids["roads"],
            linear_boundary(rm.assignments == rm.ids["roads"], 4) > 0.99,
        )
        sm.assignments[mask] = sm.ids["road"]

In [ ]:
roads = Roads()

rm.ids["roads"] = len(rm.ids)
rm.colors.append(hex_to_rgb(roads.color))
rm.radii.append(roads.radius)

In [ ]:
roads.add_region(rm)
roads.add_heights(rm, hm)
roads.add_surface(rm, hm, sm)

In [ ]:
colors = np.array(
    [hex_to_rgb(column.color) for column in sm.columns],
    dtype=np.uint8,
)
Image.fromarray(colors[sm.assignments])

### Generate the columns map

In [ ]:
COLUMN_DEPTH = 16

cm = ColumnMap(
    data=np.zeros(shape=(MAP_DIM[0], COLUMN_DEPTH, MAP_DIM[1]), dtype=int)
)

In [ ]:
assert max(len(sample) for column in sm.columns for sample in column.samples) < COLUMN_DEPTH

for i, column in enumerate(sm.columns):
    choice = smooth_buckets(MAP_DIM, len(column.samples), period=8)
    for j, sample in enumerate(column.samples):
        z, x = np.where(np.logical_and(choice == j, sm.assignments == i))
        cm.data[z, :len(sample), x] = sample

In [ ]:
colors = np.zeros(1 + max(tv.index for tv in terrain), dtype=np.uint32)
for tv in terrain:
    colors[tv.index] = tv.color
#visualize_voxels(colors[cm.data])

### Render surface with heights

In [ ]:
%%time

heights = (hm.data - hm.data.min() + COLUMN_DEPTH + 1).astype(int)
voxels = np.zeros(shape=(hm.data.shape[0], heights.max(), hm.data.shape[1]), dtype=int)

for z in range(hm.data.shape[0]):
    for x in range(hm.data.shape[1]):
        h = heights[z, x]
        voxels[z, h - 16:h, x] = cm.data[z, ::-1, x]

In [ ]:
#visualize_voxels(colors[voxels[:512, :, :512]])

### Remove padding from maps 

In [ ]:
rm.assignments = rm.assignments[CHUNK_PADDING:-CHUNK_PADDING, CHUNK_PADDING:-CHUNK_PADDING]
hm.data = hm.data[CHUNK_PADDING:-CHUNK_PADDING, CHUNK_PADDING:-CHUNK_PADDING]
sm.assignments = sm.assignments[CHUNK_PADDING:-CHUNK_PADDING, CHUNK_PADDING:-CHUNK_PADDING]
cm.data = cm.data[CHUNK_PADDING:-CHUNK_PADDING, :, CHUNK_PADDING:-CHUNK_PADDING]

### Save the maps

In [ ]:
chunk_name = "_".join([str(d) for d in np.array(CHUNK_ORIGIN) // np.array(CHUNK_DIM)])

In [ ]:
ROOT_DIR = "C:/Users/taylor/data/biomes/"

In [ ]:
with gzip.open(f"{ROOT_DIR}/alpha_world/surface/regions_{chunk_name}.dump", "wb") as f:
    pickle.dump(rm, f)

In [ ]:
with gzip.open(f"{ROOT_DIR}/alpha_world/surface/heights_{chunk_name}.dump", "wb") as f:
    pickle.dump(hm, f)

In [ ]:
with gzip.open(f"{ROOT_DIR}/alpha_world/surface/surface_{chunk_name}.dump", "wb") as f:
    pickle.dump(sm, f)

In [ ]:
with gzip.open(f"{ROOT_DIR}/alpha_world/surface/columns_{chunk_name}.dump", "wb") as f:
    pickle.dump(cm, f)